# Testing the mixed vowel substitution (`word_quota`), no speech

Runs only text → IPA → `manipulate_with_quota()` and shows:
1. how many candidate words each vowel has, and how many words could take more than one vowel
2. whether every quota was met, and that no word got more than one vowel change
3. the chosen words: spelling, original IPA → accented IPA
4. the whole text with the changes coloured by vowel
5. how the changes are spread over the sentences
6. a check over many seeds: how often each kind of word gets picked (words with several target vowels are picked more often, see the explanation in `word_quota.py`)

Nothing is synthesized. `functions.py` still loads the model at import, because the G2P lives in the same pipeline.

In [ ]:
from functions import text_to_ipa_chunks, one_sentence_chunks, check_ipa, print_stats, BOUNDARY
from word_quota import manipulate_with_quota, _word_spans
import re, random, html
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from IPython.display import display, HTML

## Settings

In [ ]:
text = (""" Elena Marchetti lavorava da sei anni nell'archivio comunale di Ponteverde, una piccola città costruita lungo un fiume nella pianura del Nord. L'archivio occupava il terzo piano di un vecchio palazzo di mattoni rossi, a pochi passi dalla piazza principale e dal municipio. Il palazzo era stato costruito alla fine dell'Ottocento, e in passato aveva ospitato una scuola e poi alcuni uffici del tribunale. La sala più grande aveva soffitti alti, scaffali di legno scuro e due lunghe finestre che guardavano verso il fiume. In quella sala si trovavano i registri delle nascite, dei matrimoni e delle proprietà, alcuni dei quali avevano più di duecento anni. Il lavoro di Elena consisteva nel catalogare i documenti, nel rispondere alle richieste dei cittadini e nel tenere in ordine le sale. Ogni mattina arrivava alle otto, accendeva le luci, apriva le finestre per qualche minuto e controllava la posta elettronica dell'ufficio. Di solito restava in archivio fino alle sette di sera, perché preferiva lavorare quando il palazzo era tranquillo e silenzioso. Nelle ore del pomeriggio gli altri uffici si svuotavano, e lei poteva concentrarsi sui registri più vecchi senza essere interrotta.
Con lei lavorava anche Paolo Ferretti, un collega più giovane che si occupava soprattutto della digitalizzazione dei documenti. Paolo passava molte ore davanti allo scanner, una grande macchina che fotografava le pagine una per una con molta precisione. Quando aveva tempo, Paolo aiutava Elena anche con le richieste dei cittadini, soprattutto quelle che arrivavano per posta elettronica. Al piano terra, invece, c'era il custode del palazzo, Aldo Rinaldi, che lavorava in quell'edificio da più di trent'anni. Aldo conosceva ogni stanza, ogni porta e ogni scala, e spesso raccontava come era cambiato il palazzo nel corso del tempo. Ogni giovedì Elena passava qualche minuto con Aldo nell'ingresso, per parlare del tempo, dei lavori in città e delle novità del comune.
All'inizio di novembre, una sera, Elena ha sentito per la prima volta un rumore strano che veniva dal soffitto della sala grande. Era una serie di piccoli colpi secchi, simili al suono di qualcuno che batte lentamente con le nocche su una porta di legno. Non assomigliava ai rumori normali di un edificio vecchio, come il vento nelle finestre o i passi delle persone nelle scale. Elena si è alzata dalla scrivania, ha camminato fino al centro della sala e ha guardato il soffitto per qualche momento. Ha pensato per un attimo a una finestra aperta, ma tutte le finestre della sala erano chiuse e il vento non era forte. Non ha visto niente di strano, perché l'intonaco era bianco e liscio come sempre, e le lampade erano ferme al loro posto. Il rumore è durato circa dieci minuti, poi si è fermato all'improvviso, e la sala è tornata silenziosa come sempre. Elena ha pensato che qualcuno stesse lavorando al piano di sopra, ma poi si è ricordata che il terzo piano era l'ultimo. Sopra l'archivio c'era soltanto il tetto, con le travi di legno e uno spazio basso dove nessuno entrava da molti anni.
La sera seguente, verso la stessa ora, il rumore è ricominciato, sempre dallo stesso punto del soffitto, vicino alla finestra. Anche questa volta è durato pochi minuti, e poi è scomparso senza lasciare nessuna traccia e nessuna spiegazione evidente. Elena ha preso un foglio e ha scritto la data, l'ora e una breve descrizione del suono, per non dimenticare i dettagli. Nei giorni successivi il rumore si è ripetuto quasi ogni sera, e qualche volta Elena lo ha sentito anche al mattino presto. Non era un suono forte, ma in una sala così silenziosa era impossibile non notarlo e non chiedersi da dove venisse. Per abitudine, Elena ha messo il foglio nel primo cassetto della scrivania, insieme alle chiavi e agli altri appunti della settimana.
Una mattina Elena ha raccontato tutto a Paolo, mentre preparavano insieme le scatole di documenti da portare allo scanner. Paolo l'ha ascoltata con attenzione e poi ha detto che, secondo lui, nel soffitto potevano esserci dei topi o altri piccoli animali. Ha spiegato che negli edifici vecchi succede spesso, soprattutto in autunno, quando gli animali cercano un posto caldo per l'inverno. Ha raccontato che anche nella casa dei suoi nonni, in campagna, una volta avevano trovato dei topi sotto il tetto. In quel caso, ha detto, gli animali facevano rumore soprattutto di notte, quando la casa era buia e tranquilla. "Se ci sono dei topi sopra la sala, dobbiamo chiamare subito qualcuno, perché potrebbero rovinare anche i documenti", ha detto Paolo. Elena ha pensato che l'idea fosse ragionevole, perché i colpi erano irregolari e potevano sembrare i movimenti di un animale. Tuttavia, le sembrava strano che gli animali facessero rumore sempre alla stessa ora, e quasi sempre per lo stesso numero di minuti. Inoltre, durante il giorno la sala era piena di rumori normali, e forse gli animali si muovevano anche in quelle ore senza farsi sentire. Ha deciso comunque di seguire il consiglio del collega, perché la protezione dei documenti era la cosa più importante per l'archivio. Nel pomeriggio ha scritto una richiesta all'ufficio tecnico del comune, chiedendo un controllo del soffitto e dello spazio sotto il tetto.
Pochi giorni dopo è arrivato un tecnico di una ditta specializzata, con una scala, una torcia potente e una piccola telecamera. La ditta lavorava spesso per il comune, e controllava regolarmente le scuole, le palestre e gli altri edifici pubblici della città. Il tecnico è salito nello spazio sotto il tetto attraverso una botola nel corridoio, che Elena non aveva mai notato prima. Elena e Paolo hanno aspettato nel corridoio, ascoltando i passi del tecnico che si muoveva lentamente sopra le loro teste. Ogni tanto sentivano anche la sua voce, perché parlava al telefono con un collega per descrivere quello che vedeva. Ha passato quasi un'ora a controllare le travi, gli angoli e il pavimento di legno sopra la sala grande dell'archivio. Ha usato la telecamera per guardare anche nei punti più stretti, dove una persona non riusciva a entrare con il corpo. Le immagini apparivano su un piccolo schermo, e il tecnico le osservava con calma prima di passare al punto successivo. Quando è sceso, ha spiegato a Elena e a Paolo che non aveva trovato nessun segno della presenza di animali. Non c'erano nidi, non c'erano escrementi e non c'erano buchi nel legno o nei muri, nemmeno vicino alla finestra. Ha aggiunto che lo spazio era molto pulito e asciutto, e che il tetto sembrava in buone condizioni in ogni punto. Prima di andare via ha lasciato alcune trappole per sicurezza, ma ha detto che probabilmente sarebbero rimaste vuote. Infatti, nelle due settimane seguenti le trappole sono rimaste vuote, mentre il rumore continuava a ripetersi quasi tutte le sere.
Un venerdì sera, mentre usciva dal palazzo, Elena si è fermata a parlare con Aldo, che stava chiudendo la porta principale. Gli ha raccontato la storia del rumore, del tecnico e delle trappole vuote, e gli ha chiesto se avesse mai sentito qualcosa di simile. Aldo ha ascoltato tutta la storia senza interrompere, come faceva sempre quando qualcuno gli parlava di un problema del palazzo. Poi ha detto che in trent'anni non aveva mai sentito rumori strani al terzo piano, né di giorno né di sera. Ha ricordato però che, all'inizio di ottobre, il comune aveva fatto sostituire tutto l'impianto di riscaldamento del palazzo. I lavori erano durati tre settimane, e durante quel periodo l'edificio era stato pieno di operai, di attrezzi e di polvere. Gli operai avevano installato una caldaia nuova e anche dei tubi nuovi, che passavano proprio sopra il soffitto dell'ultimo piano. "Forse non è un caso: il rumore è cominciato poche settimane dopo i lavori sul riscaldamento", ha detto Aldo con calma. Ha aggiunto che, se voleva, poteva darle gli orari del nuovo impianto, perché erano programmati con un timer automatico. Aldo ha spiegato che il timer accendeva e spegneva la caldaia ogni giorno alla stessa ora, senza bisogno di nessun intervento. Elena non aveva mai pensato al riscaldamento, perché i termosifoni della sala funzionavano bene e non facevano nessun rumore. Inoltre, i tubi non si vedevano, perché erano nascosti nello spazio sotto il tetto e nei muri spessi del palazzo. Tornando a casa, però, ha riflettuto sulle parole di Aldo e ha deciso di raccogliere informazioni in modo più preciso.
Dal lunedì seguente Elena ha preparato una tabella semplice, con una colonna per il giorno, una per l'ora e una per la durata. Aveva scelto di usare una tabella perché sapeva, dal suo lavoro in archivio, che i dati scritti sono più affidabili della memoria. Ogni volta che sentiva il rumore, guardava l'orologio e scriveva il momento esatto in cui il suono cominciava e finiva. Ha chiesto anche ad Aldo una copia degli orari del riscaldamento, che erano scritti su un foglio appeso vicino alla caldaia. Il foglio indicava che l'impianto si accendeva alle sei del mattino e si spegneva alle sei e mezza di sera, dal lunedì al venerdì. Nel fine settimana, invece, il riscaldamento restava spento, perché nel palazzo non lavorava nessuno e gli uffici erano chiusi. Nelle prime sere i numeri non le dicevano molto, ma dopo alcuni giorni Elena ha cominciato a vedere una certa regolarità. Per due settimane ha scritto i dati ogni giorno, anche nelle sere in cui il rumore era molto debole e difficile da sentire. Anche Paolo l'ha aiutata, perché nei giorni in cui Elena usciva prima, restava lui in sala e annotava gli orari nella tabella. Il rumore non arrivava mai nel mezzo della giornata, quando la sala era calda e il riscaldamento funzionava senza interruzioni. Una domenica è passata in archivio per prendere un libro, e ha notato che la sala era silenziosa per tutto il pomeriggio. Alla fine di due settimane ha messo la sua tabella accanto al foglio degli orari, e ha confrontato le due serie di numeri.
Il risultato era molto chiaro: il rumore cominciava sempre pochi minuti dopo lo spegnimento del riscaldamento, verso le sei e quaranta. In tutte le sere registrate, la differenza tra lo spegnimento dell'impianto e l'inizio dei colpi era sempre di circa dieci minuti. Al mattino, invece, i colpi arrivavano poco dopo l'accensione, mentre l'acqua calda cominciava a scorrere di nuovo nei tubi. Nei fine settimana, quando l'impianto restava spento, il rumore non si sentiva mai, né al mattino né alla sera. Elena ha mostrato la tabella ad Aldo, che l'ha letta con attenzione e ha confrontato ogni ora con il foglio della caldaia. Aldo ha detto che i numeri non lasciavano dubbi, e ha telefonato subito alla ditta che aveva installato il nuovo impianto. Qualche giorno dopo è arrivato un idraulico, che è salito nello spazio sotto il tetto insieme ad Aldo, con una torcia e alcuni attrezzi. Prima di salire, l'idraulico aveva guardato la tabella di Elena, e aveva detto che quei dati gli avrebbero fatto risparmiare molto tempo. Dopo pochi minuti ha trovato il punto: un tubo di metallo appoggiato direttamente su una vecchia trave di legno, vicino alla finestra. L'idraulico ha chiesto a Elena di restare nella sala grande, e poi ha battuto piano sul tubo con un piccolo attrezzo. Elena ha sentito lo stesso suono secco delle settimane precedenti, e ha confermato che veniva esattamente da quel punto del soffitto. L'idraulico ha spiegato che il metallo si allunga un poco quando diventa caldo e si accorcia di nuovo quando si raffredda. Ogni volta che la temperatura cambiava, il tubo scivolava sulla trave a piccoli scatti, e ogni scatto produceva un colpo secco. "È un fenomeno normale, che succede spesso quando i tubi nuovi vengono montati troppo vicino al legno", ha detto l'idraulico.
Per risolvere il problema, l'idraulico ha sollevato leggermente il tubo e ha inserito tra il metallo e la trave alcuni supporti di gomma. La gomma permetteva al tubo di muoversi liberamente, senza strisciare sul legno e quindi senza produrre rumore durante i cambi di temperatura. Secondo l'idraulico, quei supporti erano economici e semplici da montare, e potevano durare molti anni senza bisogno di manutenzione. Ha controllato anche gli altri tubi sopra il terzo piano, ma erano tutti montati correttamente e lontani dalle travi. L'idraulico ha spiegato che era meglio controllare tutto subito, perché lo stesso problema poteva comparire anche in altri punti. Prima di scendere, Aldo ha scritto la data dell'intervento nel quaderno dove registrava tutte le riparazioni del palazzo. Quella sera Elena è rimasta in archivio fino alle sette, come al solito, e ha ascoltato con attenzione il silenzio della sala. Alle sei e mezza il riscaldamento si è spento, ma dal soffitto non è arrivato nessun colpo, né in quel momento né più tardi. Il giorno dopo ha raccontato a Paolo come erano andate le cose, e insieme hanno tolto le trappole che erano rimaste nel corridoio. Paolo ha osservato che era meglio così, perché i documenti più antichi erano delicati e difficili da sostituire. Elena ha conservato la tabella in una cartella dell'archivio, insieme alla documentazione dei lavori sul riscaldamento del palazzo. Pensava che un giorno qualcun altro avrebbe potuto trovare utile quella informazione, magari durante altri lavori sull'edificio. Poi è tornata ai suoi registri antichi, e le settimane successive sono passate tranquille, tra le richieste dei cittadini e i documenti da catalogare.""")


text = text[:2000]          # same length as the speech notebook, or remove for the full text

quota_rule = {"kind": "word_quota",
              "targets": [{"old": "e", "new": "ə", "n": 40},     # <- your v1
                          {"old": "o", "new": "ə", "n": 45},     # <- your v2
                          {"old": "i", "new": "ɪ", "n": 50}],    # <- your v3
              "batch": 10,              # words per vowel per round
              "order": "rotate",        # or "shuffle"
              "occurrences": "all",     # or "one"
              "skip_altered_words": False}

other_rules = []            # e.g. your consonant rules, applied BEFORE the vowel quota
rules = other_rules + [quota_rule]
SEED = 1                    # same seed as RULE_SEED in the speech notebook -> the same words
N_SEEDS = 200               # seeds for the selection check at the end

## Text → IPA → manipulated IPA

In [ ]:
ipa_chunks = text_to_ipa_chunks(text)
out_chunks, masks, stats, origins = manipulate_with_quota(
    ipa_chunks, rules, return_mask=True, return_stats=True, return_origin=True, seed=SEED)

q = next(r for r in stats if r["kind"] == "word_quota")
print_stats([r for r in stats if r["kind"] != "word_quota"])
import io, contextlib
with contextlib.redirect_stdout(io.StringIO()) as _log:
    clean = check_ipa(out_chunks)
print("every symbol is in Kokoro's vocab ✓" if clean else _log.getvalue())

## 1. Candidates and overlap
A word is a candidate for a vowel if it contains that vowel. The table counts words by the **set** of target vowels they contain, so `e+o` means "can take e or o, but only one of them".

In [ ]:
TARGETS = [t if isinstance(t["old"], str) else dict(t, old=tuple(t["old"])) for t in quota_rule["targets"]]
KEYS = ["/".join(t["old"]) if isinstance(t["old"], tuple) else t["old"] for t in TARGETS]
OLDS = {k: (t["old"] if isinstance(t["old"], tuple) else (t["old"],)) for k, t in zip(KEYS, TARGETS)}

def word_targets(w):
    '''Target vowels a word contains (simple membership test, for the analysis only).'''
    return tuple(k for k in KEYS if any(o in w for o in OLDS[k]))

words_in = []                               # (chunk, index, ipa word, targets) for every word of the INPUT
for ci, s in enumerate(ipa_chunks):
    for wi, (a, b) in enumerate(_word_spans(s)):
        words_in.append((ci, wi, s[a:b], word_targets(s[a:b])))

combo = Counter("+".join(t) if t else "(none)" for *_, t in words_in)
print(f"{len(words_in)} words in the text\n")
print(f"{'contains':<14}{'words':>7}")
for k, n in sorted(combo.items(), key=lambda kv: (kv[0] == '(none)', -len(kv[0]), kv[0])):
    print(f"{k:<14}{n:>7}")
print()
for k in KEYS:
    print(f"candidates for {k}: {sum(k in t for *_, t in words_in):4d}   quota {q['quota'][k]}")

## 2. Quotas met? One change per word?

In [ ]:
ok = True
for k in KEYS:
    got, want = len(q["words"][k]), q["quota"][k]
    flag = "ok" if got == want else f"⚠ SHORT by {want - got}"
    ok &= got == want
    print(f"{k}: {got} / {want} words   {flag}")
if q["shortfall"]:
    print("shortfall:", q["shortfall"], "-> no assignment with one change per word can reach these quotas")

owner = defaultdict(list)                   # (chunk, word index) -> targets that changed it
for k in KEYS:
    for w in q["words"][k]:
        owner[(w["chunk"], w["index"])].append(k)
dup = {w: ks for w, ks in owner.items() if len(ks) > 1}
print("words changed by more than one vowel:", len(dup), "✓" if not dup else dup)
print("phonemes changed:", {k: sum(w["n_phonemes"] for w in q["words"][k]) for k in KEYS})

## 3. The chosen words
Spelling is matched by position in the sentence (letters only, so `dell'archivio` = `dell` + `archivio`). If a sentence's word counts differ between spelling and IPA, the spelling column shows `?`.

In [ ]:
sents = one_sentence_chunks(text)
ortho = [re.findall(r"[^\W\d_]+", s_) for s_ in sents]
spans_in  = [_word_spans(s) for s in ipa_chunks]
spans_out = [_word_spans(s) for s in out_chunks]
mismatch = [ci for ci in range(len(ipa_chunks)) if len(ortho[ci]) != len(spans_in[ci])]
if mismatch:
    print("spelling/IPA word counts differ in sentences:", mismatch, "\n")

def spelled(ci, wi):
    return ortho[ci][wi] if ci not in mismatch and wi < len(ortho[ci]) else "?"

def row(k, w):
    ci, wi = w["chunk"], w["index"]
    a, b = spans_in[ci][wi]; c, d = spans_out[ci][wi]
    return ci, wi, spelled(ci, wi), ipa_chunks[ci][a:b], out_chunks[ci][c:d], w["n_phonemes"]

for k in KEYS:
    rows = sorted(row(k, w) for w in q["words"][k])
    print(f"── {k} → {', '.join(sorted({w['new'] for w in q['words'][k]}))}  ({len(rows)} words)")
    print(f"   {'sent':>4} {'word':>4}  {'spelling':<16}{'original':<18}{'accented':<18}{'#changed'}")
    for ci, wi, sp, o, m, n in rows:
        print(f"   {ci:>4} {wi:>4}  {sp:<16}{o:<18}{m:<18}{n}")
    print()

## 4. The text with the changes coloured by vowel

In [ ]:
COL = ["#f59e0b", "#10b981", "#8b5cf6", "#ef4444", "#0ea5e9", "#ec4899"]
col = {k: COL[i % len(COL)] for i, k in enumerate(KEYS)}

def render(ci):
    s, m = out_chunks[ci], masks[ci]
    tgt = {wi: owner[(ci, wi)][0] for (c, wi) in owner if c == ci}
    pos2w = {}
    for wi, (a, b) in enumerate(spans_out[ci]):
        for j in range(a, b):
            pos2w[j] = wi
    ipa_html = []
    for j, ch in enumerate(s):
        e = html.escape(ch)
        wi = pos2w.get(j)
        if m[j] and wi in tgt:
            e = f'<b style="color:{col[tgt[wi]]}">{e}</b>'
        elif m[j]:
            e = f'<span style="background:#e5e7eb">{e}</span>'          # changed by another rule
        ipa_html.append(e)
    words = []
    for wi, wd in enumerate(ortho[ci]):
        k = tgt.get(wi) if ci not in mismatch else None
        words.append(f'<b style="color:{col[k]}">{html.escape(wd)}</b>' if k else html.escape(wd))
    n = Counter(tgt.values())
    tag = " ".join(f'<span style="color:{col[k]}">{k}×{n[k]}</span>' for k in KEYS if n[k])
    return (f'<div style="margin:0 0 12px"><div style="color:#888;font-size:11px">sentence {ci} {tag}</div>'
            f'<div style="font-size:13px">{" ".join(words)}</div>'
            f'<div style="font-family:monospace;font-size:13px;color:#333">{"".join(ipa_html)}</div></div>')

legend = " ".join(f'<b style="color:{col[k]}">{k}→{"|".join(sorted({w["new"] for w in q["words"][k]}))}</b>' for k in KEYS)
display(HTML(f'<div style="font-family:system-ui;max-width:1000px">{legend} '
             f'<span style="background:#e5e7eb">other rules</span><hr>'
             + "".join(render(ci) for ci in range(len(out_chunks))) + "</div>"))

## 5. Spread over the sentences

In [ ]:
per = np.zeros((len(out_chunks), len(KEYS)), int)
for (ci, wi), ks in owner.items():
    per[ci, KEYS.index(ks[0])] += 1
n_words = np.array([len(sp) for sp in spans_in])

fig, ax = plt.subplots(figsize=(10, 3))
bottom = np.zeros(len(out_chunks))
for i, k in enumerate(KEYS):
    ax.bar(range(len(out_chunks)), per[:, i], bottom=bottom, color=col[k], label=k); bottom += per[:, i]
ax.plot(range(len(out_chunks)), n_words * per.sum() / n_words.sum(), "k--", lw=1, label="expected if spread by length")
ax.set_xlabel("sentence"); ax.set_ylabel("changed words"); ax.legend(frameon=False, ncol=len(KEYS) + 1)
ax.spines[["top", "right"]].set_visible(False); plt.tight_layout(); plt.show()

for ci in range(len(out_chunks)):
    print(f"sentence {ci:2d}: {n_words[ci]:3d} words, changed " +
          "  ".join(f"{k}:{per[ci, i]:2d}" for i, k in enumerate(KEYS)) + f"   ({per[ci].sum() / n_words[ci]:.0%})")

## 6. Selection check over many seeds
For every word, how often it is changed (and by which vowel) across `N_SEEDS` seeds, grouped by the target vowels it contains.
- **P(changed)**: words with several target vowels have more ways in, so they are changed more often than words with one.
- **shared → each vowel**: for words that could take several vowels, how the changes were divided; this shows whether the batch/order setting favours one vowel.

In [ ]:
hits = defaultdict(Counter)                      # (chunk, index) -> Counter(target)
for sd in range(N_SEEDS):
    _, st = manipulate_with_quota(ipa_chunks, rules, return_stats=True, seed=sd, verbose=False)
    qq = next(r for r in st if r["kind"] == "word_quota")
    for k in KEYS:
        for w in qq["words"][k]:
            hits[(w["chunk"], w["index"])][k] += 1

groups = defaultdict(list)
for ci, wi, w, t in words_in:
    if t:
        groups["+".join(t)].append((ci, wi))
print(f"{'contains':<12}{'words':>6}{'P(changed)':>12}   changed as " + "  ".join(f"{k:>6}" for k in KEYS))
for g, ws in sorted(groups.items(), key=lambda kv: (len(kv[0]), kv[0])):
    tot = Counter()
    for w in ws:
        tot.update(hits[w])
    n = sum(tot.values())
    shares = "  ".join(f"{(tot[k] / n if n else 0):6.0%}" for k in KEYS)
    print(f"{g:<12}{len(ws):>6}{n / (len(ws) * N_SEEDS):>12.1%}   {'':>11}{shares}")

# words picked most often (should not be a handful of favourites)
freq = sorted(((sum(c.values()) / N_SEEDS, w) for w, c in hits.items()), reverse=True)[:10]
print("\nmost often changed:", ", ".join(f"{spelled(*w)}/{ipa_chunks[w[0]][slice(*spans_in[w[0]][w[1]])]} {p:.0%}" for p, w in freq))